In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk, ellipe

In [2]:
#Universal Constants
c=3*10**8
mu_0 = 4*(np.pi)*(10**-7)
epsilon = 8.854*10**-12
h_planck = 6.62607015*10**-34
hbar = (6.62607015*10**-34)/(2*np.pi)
k_b = 1.380649*10**-23
e = 1.60217663*10**-19

In [3]:
def find_effective_permeability(e_r,d,W):
    '''
    Determines the effective dielectric constant of a
    CPW when there is a single substrate
    Input:
    -   e_r = Dieletric Constant of the substrate[float] 
    -   d   = Height of substrate[m,float]
    -   W   = Width of CPW[m,float]
    Output:
    -   e_e = Effective Dieletric Constant of CPW[float]
    '''
    e_e = ((e_r+1)/(2))+((e_r-1)/(2))*(1/(np.sqrt(1+(12*d/W))))
    return e_e

def find_f0_Conformal_Mapping(w,s,e_r,d,W,l):
    '''
    Determine the resonant frequencies for a 
    CPW resonator using conformal mapping techniques.
    Determines the Inductance (L_l) and Capacitance (C_l)
    per unit length. Uses calculations to determine 
    effective dielectric constant of the CPW. 
    Input:
    -   w   = centre conductor width[m,float]
    -   s   = seperation of centre conductor[m,float]
    -   e_r = dieletric constant of substrate[float]
    -   d   = height of substrate[m,float]
    -   W   = width of CPW[m,float]
    -   l   = length of CPW[m,float]
    Output:
    -   L_l = Magnetic Inductance of CPW per unit length[,float]
    -   C_L = Capcitance of CPW per unit length[F,float]
    -   f0  = Resonant Frequency of CPW[Hz,float]
    '''
    #Find Arguements of the First Kind
    k_0 = (w)/(w+2*s)
    k_prime = np.sqrt(1-k_0**2)

    #Find Effective Permeability
    e_eff = find_effective_permeability(e_r,d,W)

    #Elleptical Integrals
    L_l = ((mu_0)/(4))*(ellipk(k_prime**2))/(ellipk(k_0**2))
    C_l = 4*epsilon*e_eff*((ellipk(k_0**2))/(ellipk(k_prime**2)))

    #Find Resonant Frequencies
    f0 = ((1/np.sqrt(L_l*C_l))/(4*l))/(10**9)

    return L_l,C_l,f0


### Error on Elleptical Integrals

The error on the arguements for the complete elliptic integral of the first kind are we follows:
$$
\frac{\mathrm{d}k_0}{\mathrm{d}w} = \frac{1}{w+2s} - \frac{w}{\left ( w+2s \right)^2} \hspace{5mm} \frac{\mathrm{d}k_0}{\mathrm{d}s} = \frac{-2w}{\left ( w+2s \right)}
$$
$$
\Delta k_0 = \sqrt{\left(\Delta w  \frac{\mathrm{d}k_0}{\mathrm{d}w} \right)^2 + \left( \Delta s \frac{\mathrm{d}k_0}{\mathrm{d}s} \right)^2}
$$
$$
\Delta k'_0 = \Delta k_0 \left | \frac{k_0}{k'_0} \right |
$$

In [4]:
def delta_k0(w,s,dw,ds):
    #Find Arguements of the First Kind
    k_0 = (w)/(w+2*s)
    k_prime = np.sqrt(1-k_0**2)

    #
    dk_0_dw = (1)/(w+2*s) - (w)/(w+2*s)**2
    dk_0_ds = -(2*w)/(w+2*s)

    dk_0 = np.sqrt((dw*dk_0_dw)**2+(ds*dk_0_ds)**2)
    dk_prime = k_0/k_prime * dk_0
    return dk_0,dk_prime

Error on the Elleptical Integrals as defined as Scipy are as follows:
$$
K(m) = \int_{0}^{\frac{\pi}{2}} \frac{1}{\sqrt{1-m \mathrm{sin}^2(\theta)}} d\theta \hspace{5mm} m = k^2
$$
$$
\frac{\mathrm{d}K}{\mathrm{d}m} = \frac{E(m) - (1-m)K(m)}{2m(1-m)}
$$
$$
\Delta K = \left | \frac{\mathrm{d}K}{\mathrm{d}m} \right |\Delta m
$$

where $E(m)$ is the complete elleptical integral of the second kind.

In [5]:
def delta_elleptical_int(w,s,dw,ds):
    #Find Arguements of the First Kind
    k_0 = (w)/(w+2*s)
    k_prime = np.sqrt(1-k_0**2)

    m_k_0 = k_0**2
    m_k_prime = k_prime**2

    dk_0,dk_prime=delta_k0(w,s,dw,ds)

    dm_k_0 = m_k_0*2*((dk_0)/(k_0))
    dm_k_prime = m_k_prime*2*((dk_prime)/(k_prime))

    #Propergate Errors
    dK_k_0 = dm_k_0*np.sqrt(((ellipe(m_k_0)-(1-m_k_0)*ellipk(m_k_0))/(2*m_k_0*(1-m_k_0)))**2)
    dK_k_prime = dm_k_0*np.sqrt(((ellipe(m_k_prime)-(1-m_k_prime)*ellipk(m_k_prime))/(2*m_k_prime*(1-m_k_prime)))**2)

    return dK_k_0, dK_k_prime



The error on the Inductance $L_{l,C}$ and Capacitance $C_{l,C}$ per unit length from conformal mapping is defined as:
$$
\Delta L_{l,C} = L_{l,c} \sqrt{\left( \frac{\Delta K (k'_0)}{K(k'_0)} \right)^2 + \left( \frac{\Delta K (k_0)}{K(k_0)} \right)^2}
$$
$$
\Delta C_{l,C} = C_{l,c} \sqrt{\left( \frac{\Delta K (k'_0)}{K(k'_0)} \right)^2 + \left( \frac{\Delta K (k_0)}{K(k_0)} \right)^2 + \left( \frac{\Delta \epsilon_{eff}}{\epsilon_{eff}} \right)^2} 
$$

In [6]:
def delta_e_eff(e_eff,e_r,d,W,dd,dW):

    d_eps_dd = -3 * (e_r - 1) / W * e_eff ** -1.5
    d_eps_dW = 3 * d * (e_r - 1) / W**2 * e_eff ** -1.5

    de_eff = np.sqrt((d_eps_dd * dd) ** 2 + (d_eps_dW * dW) ** 2)

    return de_eff



def delta_L_C_conformal_mapping(w,s,e_r,d,W,l,dw,ds,dd,dW):
    '''
     Input:
        -   w   = centre conductor width[m,float]
        -   s   = seperation of centre conductor[m,float]
        -   e_r = dieletric constant of substrate[float]
        -   d   = height of substrate[m,float]
        -   W   = width of CPW[m,float]
        -   l   = length of CPW[m,float]
        -   dw
        -   ds
        -   dd
        -   dW
    '''
    #Apply Conformal Mapping
    L_l,C_l,f0 = find_f0_Conformal_Mapping(w,s,e_r,d,W,l)

    #Find Arguements of the First Kind
    k_0 = (w)/(w+2*s)
    k_prime = np.sqrt(1-k_0**2)

    #Integrals
    K_k_0 = ellipk(k_0**2)
    K_k_prime = ellipk(k_prime**2)

    #Error on integrals
    dK_k_0,dK_k_prime = delta_elleptical_int(w,s,dw,ds)

    #Find Effective Permeability
    e_eff = find_effective_permeability(e_r,d,W)
    de_eff = delta_e_eff(e_eff,e_r,d,W,dd,dW)

    #Error on Inductance
    dL_l = L_l*np.sqrt((dK_k_0/K_k_0)**2 + (dK_k_prime/K_k_prime)**2)

    #Error on Capcitance
    dC_l = C_l*np.sqrt((dK_k_0/K_k_0)**2 + (dK_k_prime/K_k_prime)**2 + (de_eff/e_eff)**2)

    return dL_l,dC_l

In [ ]:
#Resonator Lengths
l1 = 556.1*10**-6
l2 = 636.1*10**-6
l3 = 580.1*10**-6
l4 = 668.1*10**-6
l5 = 607.1*10**-6
l6 = 538.8*10**-6
l_vals = np.array([l1,l2,l3,l4,l5,l6])
dl_vals = np.ones(len(l_vals))*0.2*10**-6

#Substrate Height
h_s = 500*10**-6
dh_s = 1*10**-6

#CPW width
w_CPW = 20*10**-6
dw_CPW = 2*10**-6

#Cryo Sapphire dielectric constant
e_sap = 10.1

#Dimensions of CPW
central_width = 10*(10**-6)
dcentral_width = 1*(10**-6)
seperation = 5*(10**-6)
dseperation = 1*(10**-6)

#Find Errors
dL_l, dC_l = delta_L_C_conformal_mapping(central_width,seperation,e_sap,h_s,w_CPW,l_vals,dcentral_width,dseperation,dh_s,dw_CPW)